In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
import time

In [2]:
# Define the state
class CrashState(TypedDict):
    input: str
    step1:str
    step2: str

In [3]:
# Define steps
def step_1(state : CrashState) -> CrashState :
  print("step1 executed")
  return {"step1":"done","input" : state["input"]}

def step_2(state : CrashState) -> CrashState :
  print(" step 2 hanging ..... Now manually interrupt from Notebook")
  time.sleep(30)   # To simulate long running hang
  return {"step2": "done"}

def step_3(state : CrashState) -> CrashState :
  print("step3 executed")
  return {"step3": "done"}


In [4]:
# Build the LangGraph
builder = StateGraph(CrashState)
builder.add_node("step_1",step_1)
builder.add_node("step_2",step_2)
builder.add_node("step_3",step_3)

builder.set_entry_point("step_1")

builder.add_edge("step_1","step_2")
builder.add_edge("step_2","step_3")
builder.add_edge("step_3",END)

checkpointer= InMemorySaver()
graph=builder.compile(checkpointer=checkpointer)


In [6]:
try:
  print("....Running Graph......Pl. manually interrupt during step 2........")
  graph.invoke({"input": "start"},config = {"configurable" : {"thread_id" : "thread-1"}})
except KeyboardInterrupt:
  print("Kernel manually interrupted............Crash simulated")



....Running Graph......Pl. manually interrupt during step 2........
step1 executed
 step 2 hanging ..... Now manually interrupt from Notebook
step3 executed


In [11]:
graph.get_state({"configurable" :{"thread_id" : "thread-1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d0487-9207-6cf5-8001-05397c943595'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-25T07:48:31.359080+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d0487-9202-6395-8000-0c94080ec21a'}}, tasks=(PregelTask(id='a5258f0c-dcd0-562b-d7b4-cc7d3ce578af', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [13]:
# Re-run to simulate Fault Tolerence
print("Running the graph again to demonstrate fault Tolerence")
final_state=graph.invoke(None,config = {"configurable" : {"thread_id" : "thread-1"}})  # Input as None as system restarted at crash point
print("\n Final State = ",final_state)              # The final step shows all steps in the workflow i.e. start -> input1 -> input2
                                                    # This proves workflow resumes from the crash point

Running the graph again to demonstrate fault Tolerence

 Final State =  {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [14]:
graph.get_state({"configurable" :{"thread_id" : "thread-1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d049d-1db9-63ef-8003-e722e65489eb'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-10-25T07:58:09.721423+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d049d-1db6-640d-8002-eb6526a62e6d'}}, tasks=(), interrupts=())